In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =========================
# CONFIG
# =========================
STORAGE_ACCOUNT = "ragadziyada"
STORAGE_KEY = "qIXjwo7EGK8an84BPCk446tqY9L7K7r3a9W2WB+voe2vUCvW1SK3qc/7UGOicKyBAtrptYcVfTB7+AStvWZi0A=="

PROCESSED_CONTAINER = "processed-p1"
CURATED_CONTAINER = "curated-p1"

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

ARTICLES_IN = f"abfss://{PROCESSED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/articles_clean/"
CUSTOMERS_IN = f"abfss://{PROCESSED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/customers_clean/"
TRANSACTIONS_IN = f"abfss://{PROCESSED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/transactions_clean/"

INTERACTIONS_OUT = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/customer_article_interactions/"
CUSTOMER_FEATURES_OUT = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/customer_features/"
ARTICLE_FEATURES_OUT = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/article_features/"
RECOMMENDATION_BASE_OUT = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/recommendation_dataset/"

# =========================
# LOAD CLEAN DATA
# =========================
articles_df = spark.read.parquet(ARTICLES_IN)
customers_df = spark.read.parquet(CUSTOMERS_IN)
transactions_df = spark.read.parquet(TRANSACTIONS_IN)

# =========================
# BUILD INTERACTIONS
# =========================
interactions = (
    transactions_df.alias("t")
    .join(customers_df.alias("c"), on="customer_id", how="left")
    .join(articles_df.alias("a"), on="article_id", how="left")
)

latest_date_row = transactions_df.select(F.max("t_dat").alias("max_date")).first()
latest_date = latest_date_row["max_date"]
print("Latest transaction date:", latest_date)

# =========================
# CUSTOMER FEATURES
# =========================
customer_features = (
    interactions.groupBy("customer_id")
    .agg(
        F.count("*").alias("customer_purchase_count"),
        F.sum("price").alias("customer_total_spend"),
        F.avg("price").alias("customer_avg_price"),
        F.countDistinct("article_id").alias("customer_unique_articles"),
        F.countDistinct("product_type_name").alias("customer_unique_product_types"),
        F.countDistinct("department_name").alias("customer_unique_departments"),
        F.max("t_dat").alias("customer_last_purchase_date"),
        F.min("t_dat").alias("customer_first_purchase_date")
    )
    .withColumn(
        "customer_recency_days",
        F.datediff(F.lit(latest_date), F.col("customer_last_purchase_date"))
    )
    .withColumn(
        "customer_purchase_frequency",
        F.when(
            F.datediff(F.col("customer_last_purchase_date"), F.col("customer_first_purchase_date")) > 0,
            F.col("customer_purchase_count") / F.datediff(F.col("customer_last_purchase_date"), F.col("customer_first_purchase_date"))
        ).otherwise(None)
    )
)

dept_pref = (
    interactions.groupBy("customer_id", "department_name")
    .count()
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("customer_id").orderBy(F.desc("count"), F.asc("department_name"))
        )
    )
    .filter("rn = 1")
    .select("customer_id", F.col("department_name").alias("customer_preferred_department"))
)

index_pref = (
    interactions.groupBy("customer_id", "index_group_name")
    .count()
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("customer_id").orderBy(F.desc("count"), F.asc("index_group_name"))
        )
    )
    .filter("rn = 1")
    .select("customer_id", F.col("index_group_name").alias("customer_preferred_index_group"))
)

basket_sizes = (
    interactions.groupBy("customer_id", "t_dat")
    .agg(F.countDistinct("article_id").alias("basket_size"))
    .groupBy("customer_id")
    .agg(F.avg("basket_size").alias("customer_avg_basket_size"))
)

customer_features = (
    customer_features
    .join(dept_pref, on="customer_id", how="left")
    .join(index_pref, on="customer_id", how="left")
    .join(basket_sizes, on="customer_id", how="left")
)

# =========================
# ARTICLE FEATURES
# =========================
article_features = (
    interactions.groupBy("article_id")
    .agg(
        F.count("*").alias("article_total_sales"),
        F.countDistinct("customer_id").alias("article_unique_customers"),
        F.avg("price").alias("article_avg_price"),
        F.max("t_dat").alias("article_last_sold_date"),
        F.expr(f"sum(case when t_dat >= date_sub(date('{latest_date}'), 7) then 1 else 0 end)").alias("article_sales_last_7d"),
        F.expr(f"sum(case when t_dat >= date_sub(date('{latest_date}'), 30) then 1 else 0 end)").alias("article_sales_last_30d"),
        F.first("product_type_name", ignorenulls=True).alias("article_product_type_name"),
        F.first("department_name", ignorenulls=True).alias("article_department_name"),
        F.first("index_group_name", ignorenulls=True).alias("article_index_group_name"),
        F.first("colour_group_name", ignorenulls=True).alias("article_colour_group_name"),
        F.first("garment_group_name", ignorenulls=True).alias("article_garment_group_name")
    )
    .withColumn(
        "article_popularity_rank",
        F.dense_rank().over(Window.orderBy(F.desc("article_total_sales")))
    )
)

# =========================
# RECOMMENDATION DATASET
# =========================
recommendation_dataset = (
    interactions
    .join(customer_features, on="customer_id", how="left")
    .join(article_features, on="article_id", how="left")
    .withColumn(
        "same_preferred_department",
        F.when(
            F.col("department_name") == F.col("customer_preferred_department"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "same_preferred_index_group",
        F.when(
            F.col("index_group_name") == F.col("customer_preferred_index_group"),
            1
        ).otherwise(0)
    )
    .withColumn("price_vs_customer_avg", F.col("price") - F.col("customer_avg_price"))
    .withColumn("is_online_channel", F.when(F.col("sales_channel_id") == 2, 1).otherwise(0))
)

# quick null check on important engineered columns
print("=== NULL CHECK: recommendation_dataset ===")
display(
    recommendation_dataset.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in [
            "customer_purchase_count",
            "customer_total_spend",
            "customer_avg_price",
            "article_total_sales",
            "article_unique_customers",
            "article_avg_price",
            "same_preferred_department",
            "same_preferred_index_group",
            "price_vs_customer_avg",
            "is_online_channel"
        ]
    ])
)
# selected final features for modeling / recommendation
selected_features = recommendation_dataset.select(
    "customer_id",
    "article_id",
    "price",
    "sales_channel_id",
    "customer_purchase_count",
    "customer_total_spend",
    "customer_avg_price",
    "customer_unique_articles",
    "customer_recency_days",
    "customer_avg_basket_size",
    "article_total_sales",
    "article_unique_customers",
    "article_avg_price",
    "article_sales_last_7d",
    "article_sales_last_30d",
    "article_popularity_rank",
    "same_preferred_department",
    "same_preferred_index_group",
    "price_vs_customer_avg",
    "is_online_channel"
)

print("selected_features rows:", selected_features.count())
display(selected_features.limit(10))


# =========================
# WRITE CURATED DATA
# =========================
interactions.write.mode("overwrite").parquet(INTERACTIONS_OUT)
customer_features.write.mode("overwrite").parquet(CUSTOMER_FEATURES_OUT)
article_features.write.mode("overwrite").parquet(ARTICLE_FEATURES_OUT)
recommendation_dataset.write.mode("overwrite").parquet(RECOMMENDATION_BASE_OUT)

print("Curated datasets written successfully.")
print("customer_features:", customer_features.count())
print("article_features:", article_features.count())
print("recommendation_dataset:", recommendation_dataset.count())

Latest transaction date: 2020-09-22
=== NULL CHECK: recommendation_dataset ===


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_purchase_count,customer_total_spend,customer_avg_price,article_total_sales,article_unique_customers,article_avg_price,same_preferred_department,same_preferred_index_group,price_vs_customer_avg,is_online_channel
0,0,0,0,0,0,0,0,0,0


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


selected_features rows: 28813419


customer_id,article_id,price,sales_channel_id,customer_purchase_count,customer_total_spend,customer_avg_price,customer_unique_articles,customer_recency_days,customer_avg_basket_size,article_total_sales,article_unique_customers,article_avg_price,article_sales_last_7d,article_sales_last_30d,article_popularity_rank,same_preferred_department,same_preferred_index_group,price_vs_customer_avg,is_online_channel
b10f43de2cdb36839eecccc7439d882a03679f74612495421486058ec5115c05,0524825010,0.025915254237288132,2,154,4.914610169491525,0.03191305304864627,149,32,5.923076923076923,1731,1601,0.04009488000469991,0,0,1763,0,1,-0.005997798811358139,1
b26630a0e48fb56ac7a7a1ef4f934a46bdf931655a824019d5723b263f551361,0677506001,0.013542372881355931,1,100,2.1833050847457622,0.02183305084745762,98,12,2.152173913043478,160,160,0.011958050847457626,0,0,3327,0,1,-0.00829067796610169,0
02d796ea767fa2e94fc6228fe70d8af1a570da973c32f7ddbe7ac747ce4a6666,0602409006,0.033440677966101694,2,107,3.4440847457627135,0.032187707904324424,100,4,7.428571428571429,44,42,0.033090138674884445,0,0,3443,0,0,0.0012529700617772699,1
af3f517549dc5fe38553173735ff2e0fe770b436b05d173b4ff5b12f8ff02769,0182909001,0.01128813559322034,1,177,5.480474576271186,0.030963133199272238,170,5,3.3207547169811322,2384,2060,0.011756291946308726,18,55,1206,0,1,-0.019674997606051896,0
b34932adfe892bed3709c0ee03d29f77a79cf5ac40c7ac24b72b5fa56b3609e0,0661794002,0.15252542372881356,2,40,1.4433898305084738,0.036084745762711846,40,26,2.857142857142857,2856,2600,0.12097633290604376,0,1,904,0,1,0.11644067796610172,1
04951f9573963f9ec689ab583f26ef7fe11bf5a1be15217e8db3b2357ab2b83e,0569929001,0.06777966101694916,2,27,0.637677966101695,0.023617702448210928,27,171,2.076923076923077,839,791,0.055346134421526846,0,0,2648,0,0,0.044161958568738224,1
06aeab63a26b9e1a247765a9005cfa48e3dd2c7326ea193cbdcb32ddcda18e98,0680187001,0.016932203389830508,2,55,1.2435593220338979,0.022610169491525417,55,24,5.5,52,51,0.014703063885267273,0,0,3435,0,0,-0.005677966101694909,1
b56cfd91d98f4182a334c728f08cb3dcdd0b0ea989b84ffadf542fbbe7a8b855,0501722022,0.022016949152542376,1,67,1.652338983050847,0.024661775866430553,64,29,2.576923076923077,160,155,0.03256588983050847,0,0,3327,0,0,-0.002644826713888177,0
b5fd4a5a95707dabe83d626a9a1065fd4ec5a95159823e8f3eed2f91b583d367,0156231001,0.005627118644067797,1,25,0.5103898305084745,0.020415593220338982,24,48,1.8461538461538463,16698,10252,0.006770968206889685,109,641,19,0,1,-0.014788474576271186,0
029ceb992cb63df03c109790046e3fdebfce0b63c968823dd461b7f18ecc6b30,0656719005,0.0501864406779661,2,56,2.105762711864406,0.03760290556900725,53,99,3.857142857142857,2406,2123,0.04725017259112103,3,3,1191,0,1,0.012583535108958849,1


Curated datasets written successfully.
customer_features: 1362281
article_features: 104547
recommendation_dataset: 28813419
